In [1]:
# ==============================================================
# DAY 12 - USED CAR DATA PREPROCESSING
# Dataset: Day12_Used_Car_Preprocessing_Dataset.csv
# ==============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# --------------------------------------------------------------
# 1. LOAD DATASET
# --------------------------------------------------------------

df = pd.read_csv("Day12_Used_Car_Preprocessing_Dataset.csv")

print("========== ORIGINAL DATASET ==========")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

# --------------------------------------------------------------
# 2. UNDERSTAND DATASET
# --------------------------------------------------------------

print("\n========== DATASET INFORMATION ==========")
df.info()

print("\n========== STATISTICAL SUMMARY ==========")
display(df.describe().T)

print("\n========== MISSING VALUES ==========")
print(df.isnull().sum())

print("\n========== DUPLICATES ==========")
print("Duplicate records:", df.duplicated().sum())

# --------------------------------------------------------------
# 3. REMOVE DUPLICATES
# --------------------------------------------------------------

df = df.drop_duplicates().reset_index(drop=True)

print("\nShape after removing duplicates:", df.shape)

# --------------------------------------------------------------
# 4. REMOVE CAR_ID
# --------------------------------------------------------------
# Car_ID is only an identifier and is not useful for prediction.

df = df.drop(columns=["Car_ID"])

print("\nCar_ID removed.")
print("Remaining columns:")
print(df.columns.tolist())

# --------------------------------------------------------------
# 5. DEFINE FEATURES AND TARGET
# --------------------------------------------------------------

TARGET = "Resale_Price_Lakh"

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("\n========== FEATURES AND TARGET ==========")
print("Features:")
print(X.columns.tolist())
print("\nTarget:", TARGET)

# --------------------------------------------------------------
# 6. TRAIN-TEST SPLIT
# --------------------------------------------------------------
# Split BEFORE fitting preprocessing to avoid data leakage.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\n========== TRAIN TEST SPLIT ==========")
print("Training features:", X_train.shape)
print("Testing features :", X_test.shape)
print("Training target  :", y_train.shape)
print("Testing target   :", y_test.shape)

# --------------------------------------------------------------
# 7. IDENTIFY COLUMN TYPES
# --------------------------------------------------------------

numeric_features = [
    "Year",
    "Mileage_Km",
    "Engine_CC",
    "Power_BHP",
    "Previous_Owners",
    "Accidents_Reported",
    "Service_Score"
]

# Condition has a natural order:
# Poor < Fair < Good < Very Good < Excellent

ordinal_features = [
    "Condition"
]

nominal_features = [
    "Brand",
    "Fuel_Type",
    "Transmission",
    "City",
    "Seller_Type"
]

print("\n========== NUMERICAL FEATURES ==========")
print(numeric_features)

print("\n========== ORDINAL FEATURE ==========")
print(ordinal_features)

print("\n========== NOMINAL FEATURES ==========")
print(nominal_features)

# --------------------------------------------------------------
# 8. OUTLIER DETECTION USING IQR
# --------------------------------------------------------------
# IQR limits are calculated ONLY from training data.

print("\n========== OUTLIER REPORT ==========")

outlier_report = []

for col in numeric_features:

    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    count = (
        (X_train[col] < lower) |
        (X_train[col] > upper)
    ).sum()

    outlier_report.append([
        col,
        Q1,
        Q3,
        IQR,
        lower,
        upper,
        count
    ])

outlier_df = pd.DataFrame(
    outlier_report,
    columns=[
        "Feature",
        "Q1",
        "Q3",
        "IQR",
        "Lower_Bound",
        "Upper_Bound",
        "Outlier_Count"
    ]
)

display(outlier_df)

# --------------------------------------------------------------
# 9. HANDLE OUTLIERS USING IQR CLIPPING
# --------------------------------------------------------------
# We clip extreme values instead of deleting complete records.
# This preserves useful used-car information.

X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

iqr_limits = {}

for col in numeric_features:

    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    iqr_limits[col] = (lower, upper)

    # Apply limits to training data
    X_train_clean[col] = X_train_clean[col].clip(
        lower=lower,
        upper=upper
    )

    # Apply SAME limits to testing data
    X_test_clean[col] = X_test_clean[col].clip(
        lower=lower,
        upper=upper
    )

print("\nIQR outlier handling completed.")

# --------------------------------------------------------------
# 10. PREPROCESSING PIPELINES
# --------------------------------------------------------------

# Numerical:
# Missing values -> Median
# Scaling -> StandardScaler

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Ordinal:
# Poor -> 0
# Fair -> 1
# Good -> 2
# Very Good -> 3
# Excellent -> 4

ordinal_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "ordinal_encoder",
            OrdinalEncoder(
                categories=[[
                    "Poor",
                    "Fair",
                    "Good",
                    "Very Good",
                    "Excellent"
                ]]
            )
        )
    ]
)

# Nominal:
# One-Hot Encoding

nominal_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

# --------------------------------------------------------------
# 11. COMBINE ALL PREPROCESSING
# --------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numeric_pipeline,
            numeric_features
        ),
        (
            "ordinal",
            ordinal_pipeline,
            ordinal_features
        ),
        (
            "nominal",
            nominal_pipeline,
            nominal_features
        )
    ]
)

# --------------------------------------------------------------
# 12. FIT ONLY ON TRAINING DATA
# --------------------------------------------------------------
# This prevents data leakage.

X_train_processed = preprocessor.fit_transform(
    X_train_clean
)

# Transform test data using the already fitted transformer
X_test_processed = preprocessor.transform(
    X_test_clean
)

print("\n========== PREPROCESSING ==========")
print("Training processed shape:", X_train_processed.shape)
print("Testing processed shape :", X_test_processed.shape)

# --------------------------------------------------------------
# 13. GET PROCESSED FEATURE NAMES
# --------------------------------------------------------------

feature_names = preprocessor.get_feature_names_out()

feature_names = [
    name.replace("numerical__", "")
        .replace("ordinal__", "")
        .replace("nominal__", "")
    for name in feature_names
]

print("\nNumber of processed features:", len(feature_names))

# --------------------------------------------------------------
# 14. CREATE PROCESSED DATAFRAMES
# --------------------------------------------------------------

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

# Add target
train_processed = X_train_processed_df.copy()
train_processed[TARGET] = y_train.reset_index(drop=True)

test_processed = X_test_processed_df.copy()
test_processed[TARGET] = y_test.reset_index(drop=True)

# --------------------------------------------------------------
# 15. DISPLAY PROCESSED DATA
# --------------------------------------------------------------

print("\n========== PROCESSED TRAINING DATA ==========")
display(train_processed.head())

print("\n========== PROCESSED TESTING DATA ==========")
display(test_processed.head())

# --------------------------------------------------------------
# 16. VERIFY MISSING VALUES
# --------------------------------------------------------------

print("\n========== MISSING VALUES AFTER PREPROCESSING ==========")

print(
    "Training missing values:",
    train_processed.isnull().sum().sum()
)

print(
    "Testing missing values:",
    test_processed.isnull().sum().sum()
)

# --------------------------------------------------------------
# 17. VERIFY DATA TYPES
# --------------------------------------------------------------

print("\n========== DATA TYPES ==========")
print(train_processed.dtypes.value_counts())

# --------------------------------------------------------------
# 18. VERIFY SCALING
# --------------------------------------------------------------

print("\n========== SCALING VERIFICATION ==========")

scaled_numeric = train_processed[
    numeric_features
]

display(
    scaled_numeric.describe().T[
        ["mean", "std", "min", "max"]
    ]
)

# --------------------------------------------------------------
# 19. FINAL DATASET
# --------------------------------------------------------------

combined_processed = pd.concat(
    [train_processed, test_processed],
    ignore_index=True
)

print("\n========== FINAL PREPROCESSED DATASET ==========")
print("Shape:", combined_processed.shape)

display(combined_processed.head())

# --------------------------------------------------------------
# 20. SAVE FILES
# --------------------------------------------------------------

train_processed.to_csv(
    "Used_Car_Preprocessed_Train.csv",
    index=False
)

test_processed.to_csv(
    "Used_Car_Preprocessed_Test.csv",
    index=False
)

combined_processed.to_csv(
    "Used_Car_Preprocessed_Dataset.csv",
    index=False
)

print("\n========== FILES SAVED ==========")
print("Used_Car_Preprocessed_Train.csv")
print("Used_Car_Preprocessed_Test.csv")
print("Used_Car_Preprocessed_Dataset.csv")

# --------------------------------------------------------------
# 21. FINAL SUMMARY
# --------------------------------------------------------------

print("\n================================================")
print("       DAY 12 PREPROCESSING COMPLETED")
print("================================================")
print("Original records       :", len(df))
print("Training records       :", len(X_train))
print("Testing records        :", len(X_test))
print("Target variable        :", TARGET)
print("Outlier method         : IQR")
print("Outlier handling       : Clipping")
print("Ordinal encoding       : Condition")
print("Nominal encoding       : One-Hot Encoding")
print("Feature scaling        : StandardScaler")
print("Data leakage prevented : YES")
print("================================================")

========== ORIGINAL DATASET ==========
Shape: (320, 15)

First 5 rows:


,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93



========== DATASET INFORMATION ==========
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Car_ID              320 non-null    object 
 1   Brand               320 non-null    object 
 2   Year                320 non-null    int64  
 3   Mileage_Km          320 non-null    int64  
 4   Engine_CC           320 non-null    int64  
 5   Power_BHP           320 non-null    float64
 6   Fuel_Type           320 non-null    object 
 7   Transmission        320 non-null    object 
 8   City                320 non-null    object 
 9   Seller_Type         320 non-null    object 
 10  Condition           320 non-null    object 
 11  Previous_Owners     320 non-null    int64  
 12  Accidents_Reported  320 non-null    int64  
 13  Service_Score       320 non-null    int64  
 14  Resale_Price_Lakh   320 non-null    float64
dtypes: float64(2),

,count,mean,std,min,25%,50%,75%,max
Year,320.0,2019.537500,3.341367,2014.0,2017.0000,2020.00,2022.000,2025.0
Mileage_Km,320.0,74110.203125,38885.260771,700.0,46323.2500,72718.50,97951.500,320000.0
Engine_CC,320.0,1346.703125,543.408160,600.0,1004.7500,1303.00,1635.250,5000.0
Power_BHP,320.0,150.489688,36.665353,51.4,128.4500,150.75,171.475,390.0
Previous_Owners,320.0,1.668750,0.865369,1.0,1.0000,1.00,2.000,4.0
Accidents_Reported,320.0,0.243750,0.528164,0.0,0.0000,0.00,0.000,2.0
Service_Score,320.0,76.203125,12.745864,55.0,64.7500,77.00,87.000,98.0
Resale_Price_Lakh,320.0,4.963031,3.359259,1.2,2.2775,4.61,6.835,28.5



========== MISSING VALUES ==========
Car_ID                0
Brand                 0
Year                  0
Mileage_Km            0
Engine_CC             0
Power_BHP             0
Fuel_Type             0
Transmission          0
City                  0
Seller_Type           0
Condition             0
Previous_Owners       0
Accidents_Reported    0
Service_Score         0
Resale_Price_Lakh     0
dtype: int64

========== DUPLICATES ==========
Duplicate records: 0

Shape after removing duplicates: (320, 15)

Car_ID removed.
Remaining columns:
['Brand', 'Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type', 'Condition', 'Previous_Owners', 'Accidents_Reported', 'Service_Score', 'Resale_Price_Lakh']

========== FEATURES AND TARGET ==========
Features:
['Brand', 'Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type', 'Condition', 'Previous_Owners', 'Accidents_Reported', 'Service_Score']

Target: Resale_Pr

...


========== MISSING VALUES AFTER PREPROCESSING ==========
Training missing values: 0
Testing missing values: 0

========== DATA TYPES ==========
float64    38
Name: count, dtype: int64

========== SCALING VERIFICATION ==========


,mean,std,min,max
Year,-3.816392e-17,1.001959,-1.698227,1.634322
Mileage_Km,1.040834e-17,1.001959,-2.046919,2.877317
Engine_CC,3.469447e-18,1.001959,-1.630394,2.799136
Power_BHP,-7.077672e-16,1.001959,-2.597881,2.616712
Previous_Owners,-4.163336e-17,1.001959,-0.755752,2.344768
Accidents_Reported,0.000000e+00,0.000000,0.000000,0.000000
Service_Score,-6.938894e-18,1.001959,-1.738645,1.713556



========== FINAL PREPROCESSED DATASET ==========
Shape: (320, 38)


,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Condition,Brand_Honda,Brand_Hyundai,...,City_Hyderabad,City_Jaipur,City_Kochi,City_Lucknow,City_Mumbai,City_Pune,Seller_Type_Certified Dealer,Seller_Type_Dealer,Seller_Type_Individual,Resale_Price_Lakh
0,-0.486391,-0.225883,-0.322882,0.304485,-0.755752,0.0,-0.454105,3.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,4.26
1,0.725445,0.089371,-0.656431,-0.330444,-0.755752,0.0,-0.614672,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,5.30
2,-1.395268,1.115798,-0.120529,0.420784,-0.755752,0.0,-0.534389,2.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.23
3,1.028404,-0.195076,0.479859,0.414498,0.484456,0.0,0.188165,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,7.09
4,-1.092309,0.706680,0.786724,-0.437314,0.484456,0.0,0.107881,3.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.69



========== FILES SAVED ==========
Used_Car_Preprocessed_Train.csv
Used_Car_Preprocessed_Test.csv
Used_Car_Preprocessed_Dataset.csv

       DAY 12 PREPROCESSING COMPLETED
Original records       : 320
Training records       : 256
Testing records        : 64
Target variable        : Resale_Price_Lakh
Outlier method         : IQR
Outlier handling       : Clipping
Ordinal encoding       : Condition
Nominal encoding       : One-Hot Encoding
Feature scaling        : StandardScaler
Data leakage prevented : YES
